# 영섭: 2-3 분류 + ESG 분류 + 2-4 감성 점수 한 번에 (아영 정리본 → news_monthly)

**준비**: 런타임 → 런타임 유형 변경 → **T4 GPU**. 드라이브에 ① 아영 `articles_clean.csv` (11/17 보정본, 129만 건) ② `company_list_최종.csv`
**순서**: 위에서부터 차례로 실행. 첫 셀의 경로 2개만 바꾸기.
**결과** (원문 없음 → 공유 OK): `news_monthly_200_scored.csv`, `news_coverage_200.csv`, `article_scores_200.csv`
**시간**: 분류 수 분 + 점수 (T4로 수십만~100만 건이면 대략 30분~1시간, 진행률이 찍힘)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CLEAN = '/content/drive/MyDrive/비정형데이터(빅카인즈)/articles_clean.csv'   # ← 아영 11/17 보정 정리본 (경로가 다르면 왼쪽 폴더 아이콘에서 파일 오른쪽 클릭 → 경로 복사)
CL    = '/content/drive/MyDrive/4팀/company_list_최종.csv'                 # ← 회사 목록 경로
OUT   = '/content/drive/MyDrive/4팀_뉴스결과'
import os, time, pandas as pd, numpy as np; os.makedirs(OUT, exist_ok=True)
print(os.path.exists(CLEAN), os.path.exists(CL))
!pip -q install transformers

## 1. 함수 불러오기 (고치지 말고 실행만)
분류 규칙 v2 (사람 정답 100건 반영) + 읽기·점수 함수

In [ ]:
"""2-3 기사 유형 분류: 키워드 규칙 v2 (11/16, 사람 정답 100건 오류 분석 반영. v1은 news_rules_v1.py)

v2에서 바꾼 것 (모두 '규칙의 버그'로 볼 수 있는 것만 고침 — 특정 기사에 맞추는 키워드 추가는 하지 않음):
 1. 증권사 리포트 꼬리표를 넓게 잡음: '-대신證', '-현대차', '-메리츠', '..대신' → 기타
 2. 목표가·목표주가·투자의견이 있으면 실적 단어가 있어도 기타 (애널리스트 전망이지 회사 공시가 아님)
 3. 키워드를 찾기 전에 회사 이름을 지움: '현대산업개발'의 '개발'이 화제형 키워드로 잡히던 문제
 4. '출연'(방송 출연까지 잡힘) → '출연금', '株'는 강한 기타에서 뺌, '어닝서프라이즈/어닝쇼크' 공시형 추가

판단 근거: 제목 + 본문 앞 200자 (팀 분류 지시문과 같음). 주가·수익률·전망 정보는 판단에 쓰지 않는다.
순서: [기타-시황/리포트 먼저 거름] → 광고 → 공시형 → 논란형 → 화제형 → 기타
복합: 두 유형 이상 키워드가 걸리면 앞 순서로 정하고 복합=Y.
      단, 공시형 + 논란형이 같이 걸리면 논란형 (교수님 예: "유상증자가 논란" → 논란형).
제목에서 먼저 찾고, 제목에 아무것도 없을 때만 본문 앞 200자에서 찾는다.
"""
import re

# 주가·시황·리포트·칼럼: 수익률 정보가 섞이므로 '기타'로 먼저 거른다
OTHER_FIRST = [
    "목표주가", "목표가", "투자의견", "매수 유지", "매수의견", "주가", "시황", "증시", "코스피", "코스닥",
    "특징주", "종목", "펀드", "ETF", "배당주", "수익률", "상한가", "하한가", "신고가", "급등", "급락", "약세", "강세",
    "칼럼", "기고", "사설", "[마켓인사이트]", "HOT종목", "주간", "전망",
    "배당주", "고배당", "투자스토리", "개미", "기관 외인", "株", "보수 1위", "사볼까", "재테크",
]
# 제목이 이 모양이면 증권사 리포트 → 기타  예: "KT&G, ... 시동"-하이투자
REPORT = re.compile(r"(?:[-–]|\.\.)\s*[^\s-]*(證|증권|투자|대우|현대|대신|동부|하이|흥국|KTB|KB|유진|신영|미래|한투|NH|SK|IBK|키움|메리츠|이베스트|하나|신한|교보|삼성|한화|동양|CJ|굿모닝|한누리|CSFB|맥쿼리|씨티|UBS|모건|골드만|노무라|다이와|CLSA|HMC|부국|한양)\s*$")
KW = {
    "광고": [
        "[광고]", "[AD]", "[PR]", "협찬", "기획광고",
        "사회공헌", "기부", "후원", "봉사", "장학", "상생", "기금", "출연금", "성금", "나눔", "지원금", "복지",
        "수상", "표창", "시상", "대상 수상", "인증 획득", "인증을 획득", "선정돼", "으뜸기업",
        "공모전", "모집", "채용", "캠페인", "페스티벌", "콘서트", "전시", "공연", "메세나", "문화예술",
        "ESG 경영", "탄소중립", "친환경 활동", "플로깅", "임직원",
        "브랜드 대상", "브랜드경쟁력", "경쟁력지수", "올해의 브랜드", "우수기업", "인증", "소비자 선정", "광고대상",
    ],
    "공시형": [
        "공시", "잠정", "실적", "어닝서프라이즈", "어닝쇼크", "어닝 서프라이즈", "영업이익", "영업익", "매출", "순이익", "순익", "흑자", "적자", "분기",
        "배당", "자사주", "주주환원", "유상증자", "무상증자", "증자", "감자", "회사채", "채권 발행", "전환사채", "CB 발행", "BW",
        "수주", "공급계약", "단일판매", "계약 체결", "주총", "주주총회", "이사회", "사장 후보", "대표이사 선임", "선임",
        "영업정지", "판매 중단", "사업 중단", "액면분할", "신규 상장", "상장폐지", "합병 결정", "분할 결정",
    ],
    "논란형": [
        "횡령", "배임", "산재", "불매", "소송", "피소", "환경오염", "오염", "압수수색", "검찰", "경찰", "수사", "구속",
        "영장", "기소", "의혹", "과징금", "제재", "적발", "리콜", "결함", "파업", "노조", "갑질", "담합", "유출",
        "사고", "화재", "폭발", "사망", "논란", "고발", "징계", "벌금", "공정위", "부당", "불공정", "비자금", "분식", "탈세",
    ],
    "화제형": [
        "출시", "신제품", "신사업", "론칭", "런칭", "선보", "제휴", "협업", "협약", "MOU", "업무협약", "파트너",
        "진출", "수출", "해외", "법인 설립", "개발", "특허", "기술", "인수", "매각", "합작", "투자 유치", "지분 투자", "설비 투자", "컨소시엄",
        "우선협상", "사업자 선정", "착공", "준공", "가동", "점유율", "돌파", "최초", "확대",
    ],
}
ORDER = ["광고", "공시형", "논란형", "화제형"]


_COMPANY = None


def _mask_company(t):
    """회사 이름(company_list_최종.csv)을 □로 지움 → 이름 속 낱말(개발, 산업, 화재, 생명…)이 키워드로 잡히지 않게"""
    global _COMPANY
    if _COMPANY is None:
        import os
        try:
            import pandas as pd
            f = os.path.join(os.path.dirname(os.path.abspath(__file__)), "company_list_최종.csv")
            names = pd.read_csv(f, dtype=str).name.dropna().unique().tolist()
            names = sorted([n for n in names if len(n) >= 3], key=len, reverse=True)
            _COMPANY = re.compile("|".join(map(re.escape, names))) if names else False
        except Exception:
            _COMPANY = False
    return _COMPANY.sub(lambda m: "□" * len(m.group(0)), t) if _COMPANY else t


def _hits(text):
    return {t: [k for k in KW[t] if k in text] for t in ORDER}


def classify(title, body=""):
    """반환: (type, 복합 'Y'/'N', 근거 키워드)"""
    title = str(title or ""); body = str(body or "")[:200]
    if REPORT.search(title):
        return "기타", "N", "증권사 리포트"
    title = _mask_company(title); body = _mask_company(body)
    strong = ["목표가", "목표주가", "투자의견", "배당주", "고배당", "펀드", "투자스토리", "개미", "기관 외인", "사볼까", "칼럼"]
    if any(k in title for k in strong):
        return "기타", "N", next(k for k in strong if k in title)
    if any(k in title for k in OTHER_FIRST) and not any(k in title for k in KW["논란형"] + ["공시", "잠정", "실적", "어닝서프라이즈", "어닝쇼크", "어닝 서프라이즈", "배당", "자사주", "판매 중단", "영업정지"]):
        return "기타", "N", next(k for k in OTHER_FIRST if k in title)
    h = _hits(title)
    if not any(h.values()):
        h = _hits(body)
    got = [t for t in ORDER if h[t]]
    if not got:
        return "기타", "N", ""
    multi = "Y" if len(got) > 1 else "N"
    t = got[0]
    if "공시형" in got and "논란형" in got:      # 공시 사실이 논란의 대상 → 논란형
        t = "논란형"
    if t == "광고" and "논란형" in got:           # 홍보성 문구가 있어도 사건이면 논란형
        t = "논란형"
    return t, multi, h[t][0]


In [ ]:
# 회사 이름을 지우고 키워드를 찾도록 (예: '현대산업개발'의 '개발'이 화제형으로 잡히지 않게)
_names = pd.read_csv(CL, dtype=str).name.dropna().unique().tolist()
_names = sorted([n for n in _names if len(n) >= 3], key=len, reverse=True)
_COMPANY = re.compile('|'.join(map(re.escape, _names)))
print('회사 이름', len(_names), '개 / 시험:', classify('현대산업개발, 고덕아이파크 1430억원 조달'))

In [ ]:
"""영섭 담당: [1] 100건 뽑기, [3] 일치율, 2-4 감성 점수 — 아영 articles_clean 파일 형식에 자동으로 맞춤"""
import os, re, pandas as pd, numpy as np

CAND = {"id": ["id", "nid", "news_id", "뉴스 식별자", "뉴스식별자"],
        "code": ["code", "종목코드", "stock_code", "ticker"],
        "company": ["company", "회사명", "name", "company_name", "회사"],
        "date": ["date", "일자", "날짜"],
        "outlet": ["outlet", "언론사", "press"],
        "title": ["title", "제목"],
        "body": ["body", "본문", "content"]}


def load_clean(path):
    """아영 articles_clean*.csv → 표준 칸(id, code, company, date, outlet, title, body, year)"""
    d = pd.read_csv(path, dtype=str) if path.endswith(".csv") else pd.read_excel(path, dtype=str)
    out = {}
    for k, cs in CAND.items():
        c = next((c for c in cs if c in d.columns), None)
        out[k] = d[c] if c else pd.Series([""] * len(d))
    x = pd.DataFrame(out)
    if (x.id == "").all():
        x["id"] = [f"r{i}" for i in range(len(x))]
    x["code"] = x.code.str.replace(r"\.0$", "", regex=True).str.zfill(6)
    x["date"] = pd.to_datetime(x.date.str[:10].str.replace(r"^(\d{4})(\d{2})(\d{2})$", r"\1-\2-\3", regex=True), errors="coerce")
    if x.date.isna().all() and "month" in d.columns:      # 날짜 칸이 없으면 month(YYYY-MM)로
        x["date"] = pd.to_datetime(d["month"].str[:7] + "-15", errors="coerce")
    x["year"] = x.date.dt.year
    missing = [k for k in ["code", "date", "title"] if (out[k] == "").all()]
    print(f"읽은 줄 {len(x):,} | 칸 연결: " + ", ".join(f"{k}←{next((c for c in CAND[k] if c in d.columns), '없음')}" for k in CAND))
    if missing:
        print("⚠ 꼭 필요한 칸이 없음:", missing)
    return x


PERIODS = [(2001, 2005), (2006, 2010), (2011, 2015), (2016, 2021)]


def draw_100(x, per_period=25, max_per_company=3, seed=42, exclude_ids=()):
    """기간별 25건, 회사당 최대 3건, 시드 고정. exclude_ids = 지난 시험에 쓴 기사 id"""
    pool = x[~x.id.isin(set(exclude_ids)) & x.year.notna()]
    pool = pool.drop_duplicates(["title", "date"])   # 같은 기사가 여러 회사에 붙은 경우 하나만
    pool = pool.sample(frac=1, random_state=seed)
    picked, cnt = [], {}
    for a, b in PERIODS:
        p = pool[(pool.year >= a) & (pool.year <= b)]
        n = 0
        for i, r in p.iterrows():
            key = r.code if r.code.strip("0") else r.company
            if cnt.get(key, 0) >= max_per_company:
                continue
            picked.append(i); cnt[key] = cnt.get(key, 0) + 1; n += 1
            if n == per_period:
                break
        if n < per_period:
            print(f"⚠ {a}–{b}: {n}건만 뽑힘 (기사 부족)")
    s = pool.loc[picked].copy()
    s.insert(0, "번호", range(1, len(s) + 1))
    print(f"뽑은 건수 {len(s)} | 기간별 {s.groupby(pd.cut(s.year, [2000, 2005, 2010, 2015, 2021])).size().tolist()} | 회사 수 {s.code.nunique()} | 회사당 최대 {s.code.value_counts().max()}")
    return s


def labeling_sheets(s, out_dir, names=("예원", "아영")):
    """사람 정답지: 컴퓨터 답 없이 2부. 열쇠 파일(id 목록)은 따로 저장"""
    os.makedirs(out_dir, exist_ok=True)
    sheet = s[["번호", "id", "company", "code", "date", "outlet", "title"]].copy()
    sheet["본문 앞 200자"] = s.body.str[:200]
    sheet["date"] = sheet.date.dt.strftime("%Y-%m-%d")
    sheet = sheet.rename(columns={"company": "회사", "code": "종목코드", "date": "날짜", "outlet": "언론사", "title": "제목"})
    sheet["정답 (광고/공시형/논란형/화제형/기타)"] = ""; sheet["복합 (Y/N)"] = ""; sheet["메모"] = ""
    paths = []
    for n in names:
        p = f"{out_dir}/분류시험_100_{n}.xlsx"; sheet.to_excel(p, index=False); paths.append(p)
    s.to_csv(f"{out_dir}/분류시험_100_원본.csv", index=False, encoding="utf-8-sig")
    print("정답지:", paths)
    return paths


VALID = ["광고", "공시형", "논란형", "화제형", "기타"]


def _norm(v):
    v = str(v).strip().replace(" ", "")
    for t in VALID:
        if v.startswith(t[:2]):
            return t
    return np.nan


def agreement(path_a, path_b, classify, out_dir, final_col=None):
    """[3] 사람1·사람2·컴퓨터 비교. 사람 둘이 다른 기사는 '최종' 칸을 영섭이 채우고 다시 실행"""
    col = "정답 (광고/공시형/논란형/화제형/기타)"
    a = pd.read_excel(path_a, dtype=str); b = pd.read_excel(path_b, dtype=str)
    m = a[["번호", "id", "회사", "제목", "본문 앞 200자", col, "복합 (Y/N)"]].merge(
        b[["번호", col, "복합 (Y/N)"]], on="번호", suffixes=("_A", "_B"))
    m["사람A"] = m[f"{col}_A"].map(_norm); m["사람B"] = m[f"{col}_B"].map(_norm)
    hh = (m.사람A == m.사람B).mean()
    m["최종"] = np.where(m.사람A == m.사람B, m.사람A, np.nan)
    fp = f"{out_dir}/분류시험_100_최종정답.xlsx"
    if os.path.exists(fp):                      # 영섭이 채운 최종 정답 반영
        f = pd.read_excel(fp, dtype=str).set_index("번호")["최종"].map(_norm)
        m["최종"] = m.번호.map(f).fillna(m["최종"])
    r = m.apply(lambda x: classify(x["제목"], x["본문 앞 200자"]), axis=1, result_type="expand")
    m[["컴퓨터", "컴퓨터_복합", "근거"]] = r
    ok = m.최종.notna()
    acc = (m.loc[ok, "컴퓨터"] == m.loc[ok, "최종"]).mean()
    tp = m[ok & (m.컴퓨터 == "화제형")]; tr = m[ok & (m.최종 == "화제형")]
    prec = (tp.최종 == "화제형").mean() if len(tp) else np.nan
    rec = (tr.컴퓨터 == "화제형").mean() if len(tr) else np.nan
    print(f"사람끼리 일치 {hh:.0%} ({(m.사람A == m.사람B).sum()}/{len(m)})")
    print(f"최종 정답이 정해진 기사 {ok.sum()}건 | 컴퓨터 일치율 {acc:.0%} (목표 80%) | 화제형 정밀도 {prec:.0%}, 재현율 {rec:.0%}")
    print(pd.crosstab(m.loc[ok, "최종"], m.loc[ok, "컴퓨터"], rownames=["사람 최종"], colnames=["컴퓨터"], margins=True))
    if not os.path.exists(fp):
        m[["번호", "회사", "제목", "사람A", "사람B", "최종"]].to_excel(fp, index=False)
        print(f"→ 사람 둘이 다른 {(~ok).sum()}건: {fp} 의 '최종' 칸을 채우고 이 셀을 다시 실행")
    m[ok & (m.컴퓨터 != m.최종)][["번호", "회사", "제목", "최종", "컴퓨터", "근거"]].to_excel(f"{out_dir}/분류시험_100_틀린것.xlsx", index=False)
    return m


# ---------- 2-4 감성 점수 ----------
def load_finbert(model="snunlp/KR-FinBert-SC"):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(model)
    mdl = AutoModelForSequenceClassification.from_pretrained(model).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; mdl.to(dev)
    if dev == "cuda": mdl.half()
    lab = {v.lower(): k for k, v in mdl.config.id2label.items()}
    print("모델", model, "| 장치", dev, "| 라벨", mdl.config.id2label)
    return tok, mdl, dev, lab


def score(texts, fb, bs=128, max_len=160):
    """s = P(긍정) − P(부정), −1~+1. 재학습 없음"""
    import torch
    tok, mdl, dev, lab = fb; out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
        with torch.no_grad():
            p = torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy()
        out.append(p[:, lab["positive"]] - p[:, lab["negative"]])
    return np.concatenate(out) if out else np.array([])


def assign_month(dates):
    """마지막 거래일(평일 근사) 당일·이후 기사는 다음 달로"""
    per = dates.dt.to_period("M")
    last = (dates + pd.offsets.BMonthEnd(0)).dt.normalize()
    return per.where(dates < last, per + 1).astype(str)


def monthly(x):
    """회사×월 집계 (원문 없음 → 밖으로 내보내도 됨)"""
    g = x.groupby(["month", "code"])
    out = g.size().rename("n_all").to_frame()
    for t, k in [("화제형", "topic"), ("공시형", "disc"), ("논란형", "contro"), ("광고", "ad")]:
        sub = x[x.type == t].groupby(["month", "code"])
        out[f"n_{k}"] = sub.size(); out[f"{k}_tone"] = sub.s.mean()
    for c in [c for c in out if c.startswith("n_")]:
        out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


## ESG 규칙 (11/16 교수님 피드백: 부정/긍정/무관, 제목 기준)

In [ ]:
"""ESG 뉴스 분류 규칙 v1 (11/16 교수님 피드백: 뉴스를 ESG 틀로 좁혀서 부정/긍정/무관 3개로만)

판정 순서 (위에서부터, 처음 걸리는 곳에서 멈춤):
 ① 주가·투자 기사(목표가, 특징주, 주가 등락, '-○○증권' 꼬리표) → 무관
 ② 회사가 주인공이 아닌 기사(스포츠단 경기, 바둑·골프 대회 협찬 경기 결과) → 무관
 ③ 부정 키워드(E·S·G 사건·사고·분쟁) → 부정
 ④ 긍정 키워드(E·S·G 좋은 활동) → 긍정
 ⑤ 나머지(실적, 공시, 신제품, 인사 등) → 무관
부정을 긍정보다 먼저 보는 이유: "사고 수습 위해 성금" 같은 기사는 사건이 핵심이라서.
제목에서만 찾는다 (v1 시험: 본문까지 보면 농구 기사 본문의 '과로', 실적 기사 본문의 '기부'처럼 엉뚱한 기사가 많이 걸림 → 정밀도 우선).
반환: (판정 '부정'/'긍정'/'무관', 영역 'E'/'S'/'G'/'', 근거 키워드)
"""
import re

SKIP_INVEST = re.compile(r"ESG\s*지수|ESG\s*펀드|ESG\s*ETF|목표가|목표주가|투자의견|특징주|주가|급등|급락|상한가|하한가|신고가|신저가|추천종목|관심주|"
                         r"[-–]\s*[^\s-]*(證|증권|투자)\s*$")
SKIP_SPORTS = re.compile(r"감독|선수|구단|프로야구|프로농구|배구|축구|[0-9]+\s*대\s*[0-9]+|우승|프로기전|챔피언십|오픈\]|역전승|연패|연승")

NEG = {
    "E": ["환경오염", "토양오염", "수질오염", "오염수", "누출", "유출", "폐수", "미세먼지 배출", "배출가스 조작", "배출 조작", "불법 배출", "환경부 고발"],
    "S": ["산재", "산업재해", "중대재해", "사망사고", "근로자 사망", "노동자 사망", "추락사", "과로사", "사고", "화재", "폭발",
          "파업", "쟁의", "노사 갈등", "임금체불", "부당해고", "갑질", "폭언", "폭행", "성희롱", "성추행", "직장 내 괴롭힘",
          "리콜", "결함", "불량", "식중독", "이물질", "피해자", "소비자 피해", "개인정보 유출", "해킹", "불매", "하청 갑질", "땅콩"],
    "G": ["횡령", "배임", "비자금", "분식", "탈세", "뇌물", "로비 의혹", "불법 로비", "로비스트", "압수수색", "검찰", "경찰", "수사", "구속", "영장", "기소",
          "의혹", "과징금", "제재", "적발", "담합", "공정위", "불공정", "부당", "일감 몰아주기", "편법", "불법", "소송", "피소",
          "고소", "고발", "징계", "벌금", "경영권 분쟁", "논란"],
}
POS = {
    "E": ["친환경", "탄소중립", "탄소 감축", "온실가스 감축", "재생에너지", "신재생", "태양광 발전소", "수소 경제", "에너지 절감",
          "환경경영", "녹색경영", "그린경영", "플로깅", "나무 심기", "숲 조성", "기후"],
    "S": ["사회공헌", "기부", "후원", "봉사", "장학", "상생", "동반성장", "성금", "나눔", "기금 전달", "기금 출연", "복지",
          "이웃돕기", "일자리 창출", "채용 확대", "청년 채용", "협력사 지원", "중소기업 지원", "안전 강화", "안전경영"],
    "G": ["ESG", "윤리경영", "투명경영", "준법경영", "지배구조 개선", "사외이사 확대", "이사회 독립", "주주친화", "지속가능경영",
          "지속가능 경영", "ESG 등급", "지배구조 우수"],
}
# 단어는 맞지만 ESG 사건이 아닌 쓰임 (예: '사고 싶은', '화재보험', '부당이득' 아님에 주의)
NEG_EXCLUDE = re.compile(r"사고\s*싶|사고팔|사고(?=\s*,|\s*(외인|외국인|기관|개인)|\s*팔)|(기관|외인|외국인|개인|큰손|연기금)\s*사고|"
                         r"실적발|무기소|문화재|화재보험|화재해상|삼성화재|현대해상화재|해상화재|사고율|불량률 개선|결함 없|논란 일축|의혹 해소|"
                         r"무혐의|승소|기각")


def _find(text, table):
    for area, words in table.items():
        for w in words:
            if w in text:
                return area, w
    return None


def classify_esg(title, body=""):
    title = str(title or ""); body = str(body or "")[:200]
    if SKIP_INVEST.search(title):
        return "무관", "", "주가·투자 기사"
    if SKIP_SPORTS.search(title):
        return "무관", "", "스포츠·대회"
    if re.search(r"승소|무죄|무혐의|혐의 벗|기각", title):
        return "무관", "", "회사에 유리한 판결"
    if re.search(r"성금|기탁|기부|봉사|구호", title):          # 재난 성금 기사('서문시장 화재 1억 성금')는 사건이 아니라 사회공헌
        hit = _find(title, POS)
        if hit:
            return "긍정", hit[0], hit[1]
    hit = _find(NEG_EXCLUDE.sub(" ", title), NEG)
    if hit:
        return "부정", hit[0], hit[1]
    hit = _find(title, POS)
    if hit:
        return "긍정", hit[0], hit[1]
    return "무관", "", ""


## 2. 읽기 + 2-3 분류

In [ ]:
x = load_clean(CLEAN)
x = x[x.date.notna()].copy()
t0 = time.time()
x[['type', '복합', '근거']] = x.apply(lambda r: classify(r.title, r.body), axis=1, result_type='expand')
x[['esg', 'esg_area', 'esg_kw']] = x.title.apply(lambda t: pd.Series(classify_esg(t)))
print('ESG:', x.esg.value_counts(normalize=True).round(4).to_dict())
print(f'분류 {len(x):,}건 {time.time()-t0:.0f}초'); print(x.type.value_counts(normalize=True).round(3))

## 3. 2-4 감성 점수: 먼저 300건으로 눈 점검 → 전체

In [ ]:
fb = load_finbert()
t = x[x.type != '광고'].sample(min(300, len(x)), random_state=1).copy()
t['s'] = score((t.title.fillna('') + ' ' + t.body.fillna('').str[:200]).tolist(), fb)
print('가장 부정'); print(t.sort_values('s')[['title', 's']].head(8).to_string(index=False))
print('가장 긍정'); print(t.sort_values('s')[['title', 's']].tail(8).to_string(index=False))

In [ ]:
u = x[x.type != '광고'].drop_duplicates('id')[['id', 'title', 'body']].copy()   # 같은 기사가 여러 회사에 붙어도 점수는 한 번만
texts = (u.title.fillna('') + ' ' + u.body.fillna('').str[:200]).tolist()
S, t0, B = [], time.time(), 20000
for i in range(0, len(texts), B):
    S.append(score(texts[i:i+B], fb))
    print(f'{min(i+B, len(texts)):,}/{len(texts):,}  {time.time()-t0:.0f}초', flush=True)
u['s'] = np.concatenate(S)
x = x.drop(columns=[c for c in ['s'] if c in x]).merge(u[['id', 's']], on='id', how='left')

## 4. 월 배정 → 회사×월 → 저장

In [ ]:
x['month'] = assign_month(x.date)
x = x[x.month <= '2021-12']
g = x.groupby(['month', 'code']); mon = g.size().rename('n_all').to_frame(); mon['all_tone'] = g.s.mean()
for tp, k in [('화제형', 'topic'), ('공시형', 'disc'), ('논란형', 'contro'), ('기타', 'etc'), ('광고', 'ad')]:
    sub = x[x.type == tp].groupby(['month', 'code']); mon[f'n_{k}'] = sub.size(); mon[f'{k}_tone'] = sub.s.mean()
for c in [c for c in mon if c.startswith('n_')]: mon[c] = mon[c].fillna(0).astype(int)
for e, k in [('부정', 'esg_neg'), ('긍정', 'esg_pos')]:
    sub = x[x.esg == e].groupby(['month', 'code']); mon[f'n_{k}'] = sub.size(); mon[f'{k}_tone'] = sub.s.mean()
for c in ['n_esg_neg', 'n_esg_pos']: mon[c] = mon[c].fillna(0).astype(int)
_n = mon.n_esg_neg + mon.n_esg_pos
mon['esg_net'] = np.where(_n > 0, (mon.n_esg_pos - mon.n_esg_neg) / _n.where(_n > 0), np.nan)   # −1(부정만) ~ +1(긍정만)
mon = mon.reset_index()
mon.to_csv(f'{OUT}/news_monthly_200_scored.csv', index=False, encoding='utf-8-sig')
cov = x.assign(year=x.date.dt.year).groupby(['code', 'year']).size().unstack(fill_value=0)
cov.to_csv(f'{OUT}/news_coverage_200.csv', encoding='utf-8-sig')
x[['id', 'code', 'date', 'month', 'type', '복합', 'esg', 'esg_area', 's']].to_csv(f'{OUT}/article_scores_200.csv', index=False, encoding='utf-8-sig')
print(mon.shape, cov.shape); print(mon.describe().round(3).T[['mean', 'min', 'max']])
print('→ news_monthly_200_scored.csv, news_coverage_200.csv 두 개를 영섭 Claude에게 보내기 (원문 없음)')